In [1]:
## imports
import pandas as pd
import numpy as np
import re
import requests
import yaml


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 1. Example 1: no credentials; no wrapper

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

## 1.1 Query to pull some data

In [2]:
## using their example query of 2011 writing scores separated by gender
## based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string
## using the ( ) syntax to formulate a long
## string without linebreaks added
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')


example_naep_query


'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'

In [3]:
## use requests to call the api
naep_resp = requests.get(example_naep_query)
naep_resp
print(type(naep_resp))

## get the json contents of the response 
## here, we're assuming valid response
naep_resp_j = naep_resp.json()
naep_resp_j

## with result, turn it into a dataframe
naep_resp_d = pd.DataFrame(naep_resp_j['result'])
naep_resp_d

<Response [200]>

<class 'requests.models.Response'>


{'status': 200,
 'serviceVersion': '6.4.2026.1',
 'dwellTimeMS': '15.6497',
 'avgWebHostCPUTotalLoad': 'N/A',
 'dataHitType': 'FROM_MEMORY',
 'Source': 'B11A',
 'result': [{'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '1',
   'varValueLabel': 'Male',
   'value': 139.099504632971,
   'isStatDisplayable': 1,
   'errorFlag': 0},
  {'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '2',
   'varValueLabel': 'Female',
   'value': 158.567104984955,
   'isStatDispl

,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0


## 1.2 What happens if there's an error in our query?

In [4]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [5]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

<Response [400]>

In [56]:
## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
# naep_wrong_resp.json() 

### 1.2.2 More all-purpose way of allowing remainder of calls to run: try, except

In [7]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except Exception as e:
    print('Failed to get result from API due to error:')
    print(e) # or just: pass

Failed to get result from API due to error:
Invalid control character at: line 1 column 293 (char 292)


### 1.2.3 Can usually also find more targeted way but that varies more across APIs

In [8]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along those lines
naep_wrong_resp.text
naep_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2733\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2348\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 947\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 354"}'

'{"status":200,"serviceVersion":"6.4.2026.1","dwellTimeMS":"15.6497","avgWebHostCPUTotalLoad":"N/A","dataHitType":"FROM_MEMORY","Source":"B11A","result": [{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"1","varValueLabel":"Male","value":139.099504632971,"isStatDisplayable":1,"errorFlag":0},{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"2","varValueLabel":"Female","value":158.567104984955,"isStatDisplayable":1,"errorFlag":0}]}'

NAEP results not found


## Activity 1: writing a function to make multiple, sequential calls

- Say we want to pull the data for grades 4, 8, and 12
- How can we write a function that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [55]:
# your code here

def grade_data_pull(grade):

    naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade={}&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011').format(grade)
    
    naep_resp_2 = requests.get(naep_query)

    naep_resp_j_2 = naep_resp_2.json()

    naep_resp_d_2 = pd.DataFrame(naep_resp_j_2['result'])
    
    return(naep_resp_d_2)

list = [8,12]

grade_data_combined = [grade_data_pull(grade) for grade in list]

grade_data_combined


[   year sample yearSampleLabel  Cohort CohortLabel stattype subject  grade  \
 0  2011     R3            2011       2     Grade 8    MN:MN     WRI      8   
 1  2011     R3            2011       2     Grade 8    MN:MN     WRI      8   
 
    scale jurisdiction       jurisLabel variable variableLabel varValue  \
 0  WRIRP           NP  National public   GENDER           Sex        1   
 1  WRIRP           NP  National public   GENDER           Sex        2   
 
   varValueLabel       value  isStatDisplayable  errorFlag  
 0          Male  139.099505                  1          0  
 1        Female  158.567105                  1          0  ,
    year sample yearSampleLabel  Cohort CohortLabel stattype subject  grade  \
 0  2011     R3            2011       3    Grade 12    MN:MN     WRI     12   
 1  2011     R3            2011       3    Grade 12    MN:MN     WRI     12   
 
    scale jurisdiction       jurisLabel variable variableLabel varValue  \
 0  WRIRP           NP  National pub

In [9]:
"My grade is " + "14"

'My grade is 14'

In [38]:
grade = 14

'My grade is 14'

NameError: name 'f' is not defined

In [11]:
"My grade is {}".format(grade)

'My grade is 14'

# 2. Example 2: needs credentials; no wrapper

Create an account here: https://www.yelp.com/developers/v3/manage_app

In [57]:
## get the key
API_KEY = "W8Y9ThHtSftrmFGzMP7PDSY89q8cfQEPC2hmGl_fvwTqocKEFeiE-xRBjFzV3KruCSLOvGLrqm6JkjOOqj2UE4xqLA9e1smmnYOsnAhssnvrPxjyrZEpWCr-QqV0anYx"

In [63]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Burlingame,CA,94010"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()


<Response [200]>

In [65]:
## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf

{'id': 'McqNkwpeLNu191f-oWXEtg',
 'alias': 'limon-burlingame-2',
 'name': 'Limon',
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/7eAjK9ThJHzaQ3k47hTnsw/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/limon-burlingame-2?adjust_creative=ffyiRxo84emVW105uNFSGw&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=ffyiRxo84emVW105uNFSGw',
 'review_count': 2372,
 'categories': [{'alias': 'tapasmallplates', 'title': 'Tapas/Small Plates'},
  {'alias': 'peruvian', 'title': 'Peruvian'}],
 'rating': 4.0,
 'coordinates': {'latitude': 37.57949, 'longitude': -122.34541},
 'transactions': ['delivery', 'pickup', 'restaurant_reservation'],
 'price': '$$',
 'location': {'address1': '1101 Burlingame Ave',
  'address2': '',
  'address3': '',
  'city': 'Burlingame',
  'zip_code': '94010',
  'country': 'US',
  'state': 'CA',
  'display_address': ['1101 Burlingame Ave', 'Burlingame, CA 94010']},
 'phone': '+16507270050',
 'display_phone': '(650) 727-0050',
 'distance'

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,price,location,phone,display_phone,distance,business_hours,attributes
0,McqNkwpeLNu191f-oWXEtg,limon-burlingame-2,Limon,https://s3-media0.fl.yelpcdn.com/bphoto/7eAjK9...,False,https://www.yelp.com/biz/limon-burlingame-2?ad...,2372,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.0,"{'latitude': 37.57949, 'longitude': -122.34541}","[delivery, pickup, restaurant_reservation]",$$,"{'address1': '1101 Burlingame Ave', 'address2'...",+16507270050,(650) 727-0050,1516.794841,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.limonrestaurants.com...
1,EF8WoEUdt-qTZ2P-DLYYNQ,new-england-lobster-market-and-eatery-burlingame,New England Lobster Market & Eatery,https://s3-media0.fl.yelpcdn.com/bphoto/eIEvQK...,False,https://www.yelp.com/biz/new-england-lobster-m...,6123,"[{'alias': 'seafoodmarkets', 'title': 'Seafood...",4.3,"{'latitude': 37.602687, 'longitude': -122.374733}","[delivery, pickup]",$$$,"{'address1': '824 Cowan Rd', 'address2': '', '...",+16504431559,(650) 443-1559,2178.573282,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.newenglandlobster.ne...
2,KCvg--UhaPEwCCOjoPpocg,mykonos-meze-burlingame,Mykonos Meze,https://s3-media0.fl.yelpcdn.com/bphoto/_rdb06...,False,https://www.yelp.com/biz/mykonos-meze-burlinga...,1022,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.5,"{'latitude': 37.57863, 'longitude': -122.34508}","[delivery, pickup]",$$,"{'address1': '226 Lorton Ave', 'address2': '',...",+16502740835,(650) 274-0835,1600.659255,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.mykonosmeze.com/_fil...
3,oChaB9hBXe-zo7wXZtobfQ,capri-lasagneria-burlingame,Capri Lasagneria,https://s3-media0.fl.yelpcdn.com/bphoto/XhilMv...,False,https://www.yelp.com/biz/capri-lasagneria-burl...,515,"[{'alias': 'italian', 'title': 'Italian'}, {'a...",4.6,"{'latitude': 37.577467, 'longitude': -122.346297}","[delivery, pickup, restaurant_reservation]",$$,"{'address1': '231 Park Rd', 'address2': '', 'a...",+16509314512,(650) 931-4512,1588.517706,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.caprilasagneria.com/...
4,PUC0Y6gMJT7jtAJIaL435g,urban-kitchen-burlingame,Urban Kitchen,https://s3-media0.fl.yelpcdn.com/bphoto/SzZVw3...,False,https://www.yelp.com/biz/urban-kitchen-burling...,401,"[{'alias': 'tradamerican', 'title': 'American'}]",4.6,"{'latitude': 37.5795025966672, 'longitude': -1...","[delivery, pickup]",$$,"{'address1': '1152 Burlingame Ave', 'address2'...",+16503934380,(650) 393-4380,1472.130772,"[{'open': [{'is_overnight': False, 'start': '1...",{}
5,t05lz4pLguPAxyyNmSRy2A,backhaus-burlingame-burlingame,Backhaus - Burlingame,https://s3-media0.fl.yelpcdn.com/bphoto/WQ4G83...,False,https://www.yelp.com/biz/backhaus-burlingame-b...,283,"[{'alias': 'bakeries', 'title': 'Bakeries'}, {...",4.4,"{'latitude': 37.57932482264846, 'longitude': -...",[],$$,"{'address1': '261 California Dr', 'address2': ...",,,1565.625047,"[{'open': [{'is_overnight': False, 'start': '0...",{'menu_url': 'https://www.backhausbread.com/me...
6,MGAziN9xIqs3FyDZ4vzC-w,stella-burlingame,Stella,https://s3-media0.fl.yelpcdn.com/bphoto/q_frPN...,False,https://www.yelp.com/biz/stella-burlingame?adj...,1279,"[{'alias': 'italian', 'title': 'Italian'}, {'a...",4.1,"{'latitude': 37.57733, 'longitude': -122.349248}","[delivery, pickup]",$$$,"{'address1': '1448 Burlingame Ave', 'address2'...",+16503475733,(650) 347-5733,1437.807756,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://stellaburlingame.com/menu...
7,A441hpB8vFH4RiX_3XCXoA,cafe-34-burlingame,Cafe 34,https://s3-media0.fl.yelpcdn.com/bphoto/q6sZu0...,False,https://www.yelp.com/biz/cafe-34-burlingame?ad...,8,"[{'alias': 'mediterranean', 'title': 'Mediterr...",5.0,"{'latitude': 37.585925, 'longitude': -122.365206}",[],NaN,"{'address1': '1232 Broadway', 'address2': '', ...",+16507813144,(6

In [66]:
## more data-specific way of summarizing
## we're doing a simple approach and just retaining
## cols that have a simple str structure
## if doing for real, would want to extract things
def clean_yelp_json(one_biz):

    ## restrict to str cols
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)

yelp_stronly = [clean_yelp_json(one_b) for one_b in yelp_genjson['businesses']]
yelp_stronly_df = pd.concat(yelp_stronly)

yelp_stronly_df.head(7)


,id,alias,name,image_url,url,price,phone,display_phone
McqNkwpeLNu191f-oWXEtg,McqNkwpeLNu191f-oWXEtg,limon-burlingame-2,Limon,https://s3-media0.fl.yelpcdn.com/bphoto/7eAjK9...,https://www.yelp.com/biz/limon-burlingame-2?ad...,$$,+16507270050,(650) 727-0050
EF8WoEUdt-qTZ2P-DLYYNQ,EF8WoEUdt-qTZ2P-DLYYNQ,new-england-lobster-market-and-eatery-burlingame,New England Lobster Market & Eatery,https://s3-media0.fl.yelpcdn.com/bphoto/eIEvQK...,https://www.yelp.com/biz/new-england-lobster-m...,$$$,+16504431559,(650) 443-1559
KCvg--UhaPEwCCOjoPpocg,KCvg--UhaPEwCCOjoPpocg,mykonos-meze-burlingame,Mykonos Meze,https://s3-media0.fl.yelpcdn.com/bphoto/_rdb06...,https://www.yelp.com/biz/mykonos-meze-burlinga...,$$,+16502740835,(650) 274-0835
oChaB9hBXe-zo7wXZtobfQ,oChaB9hBXe-zo7wXZtobfQ,capri-lasagneria-burlingame,Capri Lasagneria,https://s3-media0.fl.yelpcdn.com/bphoto/XhilMv...,https://www.yelp.com/biz/capri-lasagneria-burl...,$$,+16509314512,(650) 931-4512
PUC0Y6gMJT7jtAJIaL435g,PUC0Y6gMJT7jtAJIaL435g,urban-kitchen-burlingame,Urban Kitchen,https://s3-media0.fl.yelpcdn.com/bphoto/SzZVw3...,https://www.yelp.com/biz/urban-kitchen-burling...,$$,+16503934380,(650) 393-4380
t05lz4pLguPAxyyNmSRy2A,t05lz4pLguPAxyyNmSRy2A,backhaus-burlingame-burlingame,Backhaus - Burlingame,https://s3-media0.fl.yelpcdn.com/bphoto/WQ4G83...,https://www.yelp.com/biz/backhaus-burlingame-b...,$$,,
MGAziN9xIqs3FyDZ4vzC-w,MGAziN9xIqs3FyDZ4vzC-w,stella-burlingame,Stella,https://s3-media0.fl.yelpcdn.com/bphoto/q_frPN...,https://www.yelp.com/biz/stella-burlingame?adj...,$$$,+16503475733,(650) 347-5733


# Activity 2: pull restaurants in a different location

- Try running a business search query for your hometown or another place by constructing a query similar to `yelp_genquery` but changing the location parameter
- Other endpoints require feeding what's called the business' fusion id into the API. Take an id from `yelp_stronly.id` and use the documentation here to pull the reviews for that business: https://docs.developer.yelp.com/reference/v3_business_reviews
- **Challenge**: generalize the previous step by writing a function that (1) takes a list of business ids as an input, (2) calls the reviews API for each id, (3) returns the results, and (4) rowbinds all results, i.e. turns them into a single, usable DataFrame

In [69]:
# your code here
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Burlingame,CA,94010"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()

def clean_yelp_json(one_biz):

    ## restrict to str cols
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)

yelp_stronly = [clean_yelp_json(one_b) for one_b in yelp_genjson['businesses']]
yelp_stronly_df = pd.concat(yelp_stronly)

yelp_stronly_df.head(7)

<Response [200]>

,id,alias,name,image_url,url,price,phone,display_phone
McqNkwpeLNu191f-oWXEtg,McqNkwpeLNu191f-oWXEtg,limon-burlingame-2,Limon,https://s3-media0.fl.yelpcdn.com/bphoto/7eAjK9...,https://www.yelp.com/biz/limon-burlingame-2?ad...,$$,+16507270050,(650) 727-0050
EF8WoEUdt-qTZ2P-DLYYNQ,EF8WoEUdt-qTZ2P-DLYYNQ,new-england-lobster-market-and-eatery-burlingame,New England Lobster Market & Eatery,https://s3-media0.fl.yelpcdn.com/bphoto/eIEvQK...,https://www.yelp.com/biz/new-england-lobster-m...,$$$,+16504431559,(650) 443-1559
KCvg--UhaPEwCCOjoPpocg,KCvg--UhaPEwCCOjoPpocg,mykonos-meze-burlingame,Mykonos Meze,https://s3-media0.fl.yelpcdn.com/bphoto/_rdb06...,https://www.yelp.com/biz/mykonos-meze-burlinga...,$$,+16502740835,(650) 274-0835
oChaB9hBXe-zo7wXZtobfQ,oChaB9hBXe-zo7wXZtobfQ,capri-lasagneria-burlingame,Capri Lasagneria,https://s3-media0.fl.yelpcdn.com/bphoto/XhilMv...,https://www.yelp.com/biz/capri-lasagneria-burl...,$$,+16509314512,(650) 931-4512
PUC0Y6gMJT7jtAJIaL435g,PUC0Y6gMJT7jtAJIaL435g,urban-kitchen-burlingame,Urban Kitchen,https://s3-media0.fl.yelpcdn.com/bphoto/SzZVw3...,https://www.yelp.com/biz/urban-kitchen-burling...,$$,+16503934380,(650) 393-4380
t05lz4pLguPAxyyNmSRy2A,t05lz4pLguPAxyyNmSRy2A,backhaus-burlingame-burlingame,Backhaus - Burlingame,https://s3-media0.fl.yelpcdn.com/bphoto/WQ4G83...,https://www.yelp.com/biz/backhaus-burlingame-b...,$$,,
MGAziN9xIqs3FyDZ4vzC-w,MGAziN9xIqs3FyDZ4vzC-w,stella-burlingame,Stella,https://s3-media0.fl.yelpcdn.com/bphoto/q_frPN...,https://www.yelp.com/biz/stella-burlingame?adj...,$$$,+16503475733,(650) 347-5733


In [70]:
url = "https://api.yelp.com/v3/businesses/new-england-lobster-market-and-eatery-burlingame/reviews?limit=20&sort_by=yelp_sort"
headers = {"accept": "application/json"}
response = requests.get(url, headers=headers)
print(response.text)

{"error": {"code": "VALIDATION_ERROR", "description": "Authorization is a required parameter.", "field": "Authorization", "instance": null}}
